# 09 Daten zurücksetzen

## Zweck
Alle erzeugten Dateien unter `data/` löschen, damit die gesamte Notebook-Pipeline (`01` bis `08`) aus einem sauberen Zustand erneut ausgeführt werden kann. Die Verzeichnisse bleiben erhalten; nur Dateien werden entfernt.

## Eingaben

Alle Dateien unter `data/bronze/`, `data/silver/`, `data/gold/`, `data/samples/` und `data/checkpoints/`.

## Ausgaben

Leere Verzeichnisstruktur unter `data/`. Es verbleiben keine Parquet-, CSV-, JSON-, HTML- oder JSONL-Dateien.
Die erneute Ausführung der Notebooks `01` bis `08` befüllt die Verzeichnisse wieder.

## Konfiguration

`DRY_RUN=true` als Umgebungsvariable oder Zellvariable zeigt vorab an, welche Dateien gelöscht würden.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
configured_data_dir = Path(os.getenv("DATA_DIR", "data"))
DATA_DIR = (configured_data_dir if configured_data_dir.is_absolute() else PROJECT_ROOT / configured_data_dir).resolve()

DRY_RUN = os.getenv("DRY_RUN", "true").lower() == "true"
ALLOW_EXTERNAL_DATA_RESET = os.getenv("ALLOW_EXTERNAL_DATA_RESET", "false").lower() == "true"

assert DATA_DIR != PROJECT_ROOT, "Das Repository-Stammverzeichnis wird nicht zurückgesetzt."
assert DATA_DIR != Path(DATA_DIR.anchor), "Das Stammverzeichnis des Dateisystems wird nicht zurückgesetzt."
is_project_data_dir = DATA_DIR == (PROJECT_ROOT / "data").resolve()
assert is_project_data_dir or ALLOW_EXTERNAL_DATA_RESET, (
    "DATA_DIR verweist auf einen Pfad außerhalb von PROJECT_ROOT/data. ALLOW_EXTERNAL_DATA_RESET=true erst nach Prüfung des Pfads setzen."
)

def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

MANAGED_DIRS = [
    DATA_DIR / "bronze" / "eea",
    DATA_DIR / "bronze" / "open_meteo_raw",
    DATA_DIR / "bronze" / "wikipedia_html",
    DATA_DIR / "bronze",
    DATA_DIR / "silver",
    DATA_DIR / "gold",
    DATA_DIR / "samples",
    DATA_DIR / "checkpoints",
    DATA_DIR,
]

print(f"PROJECT_ROOT              : {PROJECT_ROOT}")
print(f"DATA_DIR                  : {DATA_DIR}")
print(f"DRY_RUN                   : {DRY_RUN}")
print(f"ALLOW_EXTERNAL_DATA_RESET : {ALLOW_EXTERNAL_DATA_RESET}")

## Umsetzung

### Schritt 1: Zu löschende Dateien anzeigen

In [ ]:
import pandas as pd

all_files = sorted(DATA_DIR.rglob("*")) if DATA_DIR.exists() else []
file_records = [
    {
        "path": display_path(p),
        "size_bytes": p.stat().st_size,
        "type": p.suffix or "(keine Erweiterung)",
    }
    for p in all_files
    if p.is_file()
]

preview_df = pd.DataFrame(file_records) if file_records else pd.DataFrame(columns=["path", "size_bytes", "type"])
print(f"{len(preview_df)} Datei(en) werden {'VORSCHAU (DRY_RUN)' if DRY_RUN else 'gelöscht'}:")
preview_df

### Schritt 2: Alle Dateien löschen und Verzeichnisstruktur erhalten

In [ ]:
deleted = []
skipped = []

for p in sorted(DATA_DIR.rglob("*"), reverse=True) if DATA_DIR.exists() else []:
    if not p.is_file():
        continue
    if DRY_RUN:
        skipped.append(display_path(p))
    else:
        p.unlink()
        deleted.append(display_path(p))

if DRY_RUN:
    print(f"DRY RUN — {len(skipped)} Datei(en) würden gelöscht, keine Änderung.")
else:
    print(f"{len(deleted)} Datei(en) gelöscht.")

# Sicherstellen, dass alle erwarteten Verzeichnisse weiterhin vorhanden sind
for d in MANAGED_DIRS:
    d.mkdir(parents=True, exist_ok=True)

## Validierung und Qualitätsprüfungen

Bestätigen, dass unter `data/` keine Dateien verbleiben und alle erwarteten Verzeichnisse vorhanden sind.

In [ ]:
remaining_files = [p for p in DATA_DIR.rglob("*") if p.is_file()]

if not DRY_RUN:
    assert remaining_files == [], f"Noch {len(remaining_files)} Datei(en) vorhanden: {remaining_files[:5]}"

for d in MANAGED_DIRS:
    assert d.exists() and d.is_dir(), f"Verzeichnis fehlt nach Reset: {d}"

status = "DRY RUN — keine Änderung" if DRY_RUN else f"Reset abgeschlossen. {len(deleted)} Datei(en) gelöscht."
print(status)

dir_summary = pd.DataFrame(
    [{"directory": display_path(d), "exists": d.exists()} for d in MANAGED_DIRS]
)
dir_summary

## Ergebnisse

Nach der Ausführung ist `data/` leer. Die Notebooks `01` bis `08` werden in dieser Reihenfolge erneut ausgeführt, um Bronze-, Silver-, Gold- und Beispielausgaben neu zu erzeugen.

## Einschränkungen

Manuell unter `data/bronze/eea/` abgelegte reale EEA-Quelldateien werden ebenfalls gelöscht. Reale EEA-Daten müssen vor Notebook `03` erneut abgelegt werden.